# Embedding Fine-tuning with NeMo Microservices

Fine-tune an embedding model and improve retrieval by 6-10% in ~1 hour.

## Prerequisites

- **No local GPUs required:** Training and inference jobs run on the internal NeMo Microservices deployment
- **HuggingFace token:** Get one at https://huggingface.co/settings/tokens (set `HF_TOKEN` environment variable or enter when prompted during notebook run)


## Overview

Fine-tuning an embedding model on your domain data improves retrieval accuracy. In a Retrieval-Augmented Generation (RAG) pipeline, this means the LLM receives more relevant context, producing better answers. For search applications, users find what they need more often.

This notebook walks through the complete workflow: fine-tune a base embedding model on scientific paper data, deploy it as a production NVIDIA Inference Microservice (NIM), and measure the improvement.


## Objectives

By the end of this notebook, you will:
- Fine-tune [`nvidia/llama-3.2-nv-embedqa-1b-v2`](https://build.nvidia.com/nvidia/llama-3_2-nv-embedqa-1b-v2) on 65K scientific paper triplets from [SPECTER dataset](https://huggingface.co/datasets/embedding-data/SPECTER)
- Deploy the fine-tuned model as a production-ready NIM inference service
- Evaluate retrieval performance on the [SciDocs benchmark](https://huggingface.co/datasets/BeIR/scidocs)
- Achieve measurable improvement: baseline Recall@5 of 0.159 to ~0.17 (+6-10%)

**Recall@5** measures the fraction of relevant documents that appear in the top 5 search results.

*Note: To recreate the baseline (0.159), deploy the base model without fine-tuning and run evaluation.*

In [ ]:
# Install required packages
%pip install -q datasets huggingface_hub openai nemo-microservices

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore', category=Warning, module='tqdm')

import json, requests, os
from time import sleep, time
from datasets import load_dataset
from getpass import getpass
from nemo_microservices import NeMoMicroservices
from huggingface_hub import HfApi
from openai import OpenAI

In [ ]:
# Configuration
NDS_URL = "https://datastore.aire.nvidia.com"
NEMO_URL = "https://nmp.aire.nvidia.com"
NIM_URL = "https://nim.aire.nvidia.com"
EVAL_NIM_URL = "http://nemo-nim-proxy:8000"  # Internal URL for evaluation (avoids cluster SSL issues)

# Credentials - prompts if not set via env var
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HuggingFace token (https://huggingface.co/settings/tokens): ")
NAMESPACE = os.environ.get("NAMESPACE") or input("Namespace (e.g. yourname_embedding): ")

In [ ]:
# Initialize NeMo client
nemo = NeMoMicroservices(base_url=NEMO_URL, inference_base_url=NIM_URL)
print("NeMo client initialized")

## Step 1: Prepare Data

Download 10% of the SPECTER dataset containing ~684K scientific paper triplets (query, positive, negative) and format for embedding fine-tuning.

**Dataset format:** Each triplet teaches the model via contrastive learning to maximize similarity between query and positive document while minimizing similarity between query and negative document.


In [ ]:
# Download and prepare training data
DATASET_SIZE = 68400      # 10% of full dataset (684K triplets) - increase for better results
VALIDATION_SPLIT = 0.05   # 5% held out for validation

print("Downloading SPECTER dataset...")
os.environ["HF_TOKEN"] = HF_TOKEN
data = load_dataset("embedding-data/SPECTER")['train'].shuffle(seed=42).select(range(DATASET_SIZE))

print("Splitting into train/validation...")
splits = data.train_test_split(test_size=VALIDATION_SPLIT, seed=42)
train_data = splits['train']
validation_data = splits['test']

# Save as JSONL (required format for Customizer)
print("Saving to JSONL...")
os.makedirs("data", exist_ok=True)
for name, dataset in [("training", train_data), ("validation", validation_data)]:
    with open(f"data/{name}.jsonl", "w") as f:
        for row in dataset:
            f.write(json.dumps({"query": row['set'][0], "pos_doc": row['set'][1], "neg_doc": [row['set'][2]]}) + "\n")

print(f"Prepared {len(train_data):,} training, {len(validation_data):,} validation samples")
print(f"\nExample triplet:")
print(f"  Query:    {train_data[0]['set'][0][:100]}...")
print(f"  Positive: {train_data[0]['set'][1][:100]}...")
print(f"  Negative: {train_data[0]['set'][2][:100]}...")


## Step 2: Upload to NeMo

Upload the prepared data to NeMo Data Store and register it for training.


In [ ]:
# Upload to NeMo Data Store
# Note: We use the HuggingFace Hub API (huggingface_hub library) with endpoint pointed at NeMo Data Store

print("Creating namespace...")
nemo.namespaces.create(id=NAMESPACE)  # For job management
requests.post(f"{NDS_URL}/v1/datastore/namespaces", data={"namespace": NAMESPACE})  # For data storage

print("Creating repository...")
hf = HfApi(endpoint=f"{NDS_URL}/v1/hf", token=None)
hf.create_repo(f"{NAMESPACE}/data", repo_type='dataset')

# Paths must match Customizer conventions: training/training.jsonl, validation/validation.jsonl
print("Uploading files...")
hf.upload_file(path_or_fileobj="data/training.jsonl", path_in_repo="training/training.jsonl", repo_id=f"{NAMESPACE}/data", repo_type='dataset')
hf.upload_file(path_or_fileobj="data/validation.jsonl", path_in_repo="validation/validation.jsonl", repo_id=f"{NAMESPACE}/data", repo_type='dataset')

print("Registering dataset...")
nemo.datasets.create(name="data", namespace=NAMESPACE, files_url=f"hf://datasets/{NAMESPACE}/data")
print("Upload complete")


## Step 3: Train Model

Fine-tune using supervised contrastive learning (model learns to pull query-positive pairs closer while pushing query-negative pairs apart).

**Config vs Job:** A *config* defines the training template (base model, GPU settings). A *job* runs training with that config + dataset + hyperparameters.


In [ ]:
# Create training config
# - "target" = base model to fine-tune
# - "output_model" (in job below) = where fine-tuned weights are saved
BASE_MODEL = "nvidia/llama-3.2-nv-embedqa-1b@v2"
NUM_GPUS = 1
MICRO_BATCH_SIZE = 8
MAX_SEQ_LENGTH = 2048

print("Creating config...")
nemo.customization.configs.create(
    name="embedding-config@v1", 
    namespace=NAMESPACE, 
    target=BASE_MODEL,
    training_options=[{
        "training_type": "sft",
        "finetuning_type": "all_weights",  # Full fine-tuning (vs "lora" for faster training)
        "num_gpus": NUM_GPUS,
        "micro_batch_size": MICRO_BATCH_SIZE
    }], 
    max_seq_length=MAX_SEQ_LENGTH)

In [ ]:
# Start training job (~30 min)
EPOCHS = 1
BATCH_SIZE = 256
LEARNING_RATE = 5e-6

print("Starting training...")
training_job = nemo.customization.jobs.create(
    name="embedding-training", 
    config=f"{NAMESPACE}/embedding-config@v1", 
    dataset={"namespace": NAMESPACE, "name": "data"},
    hyperparameters={
        "finetuning_type": "all_weights",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE
    }, 
    output_model=f"{NAMESPACE}/embedding-model")

print(f"Job ID: {training_job.id}")

In [ ]:
# Monitor training (~30 min)
POLL_INTERVAL = 10  # Seconds between status checks
last_step = -1
while True:
    status = nemo.customization.jobs.retrieve(training_job.id)
    if status.status not in ["pending", "created", "running"]:
        break
    
    d = status.status_details
    elapsed = f"{int(d.elapsed_time)//60}m {int(d.elapsed_time)%60}s"
    
    if d.epochs_completed >= 1:
        print(f"\rSaving model... | {elapsed}", end="")
    elif d.metrics and d.metrics.metrics.train_loss:
        step = d.metrics.metrics.train_loss[-1].step
        if step != last_step:
            if last_step == -1: print()
            loss = d.metrics.metrics.train_loss[-1].value
            pct = int(step / d.steps_per_epoch * 100) if d.steps_per_epoch else 0
            print(f"{pct:3d}% | Step {step} | Loss: {loss:.4f} | {elapsed}")
            last_step = step
    else:
        print(f"\rInitializing... | {elapsed}", end="")
    sleep(POLL_INTERVAL)

print(f"\n\nTraining complete | {int(status.status_details.elapsed_time)//60}m")

## Step 4: Deploy Model

Deploy the fine-tuned model as a NIM inference service (~5 minutes).


In [ ]:
# Deploy as NIM (~5 min)
DEPLOYMENT_NAME = "embedding-nim"
NIM_IMAGE = "nvcr.io/nim/nvidia/llama-3.2-nv-embedqa-1b-v2"
NIM_IMAGE_TAG = "1.6.0"
DEPLOYMENT_GPUS = 1

print("Deploying model...")
try:
    existing = nemo.deployment.model_deployments.retrieve(deployment_name=DEPLOYMENT_NAME, namespace=NAMESPACE)
    print(f"Deployment exists (status: {existing.status_details.status})")
except:
    nemo.deployment.model_deployments.create(
        name=DEPLOYMENT_NAME,
        namespace=NAMESPACE,
        config={
            "model": f"{NAMESPACE}/embedding-model@{training_job.id}",
            "nim_deployment": {
                "image_name": NIM_IMAGE,
                "image_tag": NIM_IMAGE_TAG,
                "gpu": DEPLOYMENT_GPUS,
                "disable_lora_support": True  # Optimizes NIM when not using LoRA adapters
            }
        }
    )
    print("Created, waiting...")

In [ ]:
# Wait for deployment (~5 min)
POLL_INTERVAL = 10  # Seconds between status checks

start = time()
while True:
    deployment = nemo.deployment.model_deployments.retrieve(deployment_name=DEPLOYMENT_NAME, namespace=NAMESPACE)
    if deployment.status_details.status == 'ready':
        break
    elapsed = int(time() - start)
    print(f"\rStatus: {deployment.status_details.status} | {elapsed//60}m {elapsed%60}s", end="")
    sleep(POLL_INTERVAL)

print(f"\nDeployed | {int(time() - start)//60}m")

## Step 5: Test Inference

Verify the deployed model responds to embedding requests.

In [ ]:
# Test inference
INIT_WAIT = 30  # Seconds to wait for model to fully initialize from deployment
sleep(INIT_WAIT)

client = OpenAI(base_url=f"{NIM_URL}/v1", api_key="None")
response = client.embeddings.create(
    input=["Deep learning for computer vision"], 
    model=f"{NAMESPACE}/embedding-model", 
    extra_body={"input_type": "query"})

print(f"Inference OK | Embedding dim: {len(response.data[0].embedding)}")


## Step 6: Evaluate Performance

Run the SciDocs benchmark to measure retrieval quality (~10 minutes).

In [ ]:
# Run evaluation on SciDocs (~10 min)
TOP_K = 10  # Retrieve top 10, Recall@5 checks first 5

print("Starting evaluation...")

# Config: what benchmark to run and what metrics to compute
eval_config = {
    "type": "retriever",  # Evaluating retrieval (vs generation, classification, etc.)
    "namespace": NAMESPACE,
    "tasks": {
        "scidocs": {
            "type": "beir",  # BEIR: standard benchmark format for retrieval
            "dataset": {"files_url": "file://scidocs/"},  # Pre-loaded on cluster
            "metrics": {"recall_5": {"type": "recall_5"}}}}}

# Target: which model to evaluate and how to call it
eval_target = {
    "type": "retriever",
    "retriever": {
        "pipeline": {
            # Same model encodes both queries and documents
            "query_embedding_model": {
                "api_endpoint": {"url": f"{EVAL_NIM_URL}/v1/embeddings", "model_id": f"{NAMESPACE}/embedding-model"}},
            "index_embedding_model": {
                "api_endpoint": {"url": f"{EVAL_NIM_URL}/v1/embeddings", "model_id": f"{NAMESPACE}/embedding-model"}},
            "top_k": TOP_K}}}

eval_job = nemo.evaluation.jobs.create(config=eval_config, target=eval_target)
print(f"Job ID: {eval_job.id}")

In [ ]:
# Wait for evaluation (~10 min)
POLL_INTERVAL = 10  # Seconds between status checks

start = time()
while True:
    status = nemo.evaluation.jobs.retrieve(eval_job.id)
    if status.status not in ["pending", "created", "running"]:
        break
    elapsed = int(time() - start)
    print(f"\rRunning... {elapsed//60}m {elapsed%60}s", end="")
    sleep(POLL_INTERVAL)

print(f"\nComplete | {int(time() - start)//60}m")
results = nemo.evaluation.jobs.results(eval_job.id)

## Step 7: Results

Compare your fine-tuned model against the pretrained baseline.

In [ ]:
# Display results
BASELINE_RECALL = 0.159  # Pretrained model on SciDocs
finetuned_recall = results.tasks['scidocs'].metrics['retriever.recall_5'].scores['recall_5'].value
improvement = ((finetuned_recall / BASELINE_RECALL) - 1) * 100

print("=" * 60)
print("RESULTS: SciDocs Retrieval Benchmark")
print("=" * 60)
print(f"Metric: Recall@5 (relevant docs found in top 5 results)")
print()
print(f"Baseline (pretrained):  {BASELINE_RECALL:.3f}")
print(f"Fine-tuned model:       {finetuned_recall:.3f}")
print(f"Improvement:           +{improvement:.1f}%")
print("=" * 60)
print(f"\nEndpoint: {NIM_URL}/v1/embeddings")
print(f"Model: {NAMESPACE}/embedding-model")

## Summary

You fine-tuned `nvidia/llama-3.2-nv-embedqa-1b-v2` on 65K scientific paper triplets from SPECTER, then evaluated on the **SciDocs benchmark**.

| Model | Recall@5 | Improvement |
|-------|----------|-------------|
| Pretrained baseline | 0.159 | |
| **Your fine-tuned model** | **~0.17** | **+6-10%** |

A 6-10% improvement means your model surfaces 1-2 more relevant documents per 100 queries. In a RAG pipeline, this translates to better context for the LLM and more accurate answers. For search, users find what they need more often.

Your model is deployed and ready to use at the endpoint shown above.


## Next Steps

**Scale Up:**
- Train on full SPECTER dataset for additional improvement
- Increase to 3 epochs for better convergence

**Apply to Your Domain:**
- [Format your data as query-positive-negative triplets](https://docs.nvidia.com/nemo/microservices/latest/fine-tune/models/embedding.html#data-preparation)
- Replace SPECTER dataset with your domain data (legal, medical, product catalogs, etc.)
- Evaluate on your own retrieval tasks

**Learn More:**
- [NeMo Microservices Documentation](https://docs.nvidia.com/nemo/microservices/latest/)
- [Embedding Model Guide](https://build.nvidia.com/nvidia/llama-3_2-nv-embedqa-1b-v2)
- [Other NeMo Tutorials](../../../README.md)


## Cleanup

Uncomment cleanup cells as needed to delete resources.

In [ ]:
# Delete deployment (frees GPU, keeps model for later redeployment)
# print("Deleting deployment...")
# nemo.deployment.model_deployments.delete(deployment_name=DEPLOYMENT_NAME, namespace=NAMESPACE)
# print("Deployment deleted - GPU freed")

In [ ]:
# # Delete model (PERMANENT - must retrain to recover)
# print("Deleting model...")
# for m in nemo.models.list(filter={"namespace": NAMESPACE}).data:
#     nemo.models.delete(namespace=NAMESPACE, model_name=m.name.split('/')[-1])
# print("Model deleted")

In [ ]:
# # Delete dataset (PERMANENT)
# print("Deleting dataset...")
# nemo.datasets.delete(namespace=NAMESPACE, dataset_name="data")
# hf.delete_repo(f"{NAMESPACE}/data", repo_type='dataset')
# print("Dataset deleted")

In [ ]:
# # Delete configs (PERMANENT)
# print("Deleting configs...")
# nemo.customization.configs.delete(config_name="embedding-config@v1", namespace=NAMESPACE)
# for cfg in nemo.evaluation.configs.list(filter={"namespace": NAMESPACE}).data:
#     nemo.evaluation.configs.delete(config_id=cfg.id)
# print("Configs deleted")

In [ ]:
# UNCOMMENT TO DELETE EVAL JOBS (permanent)
# print("Deleting eval jobs...")
# for j in nemo.evaluation.jobs.list(filter={"namespace": NAMESPACE}).data:
#     nemo.evaluation.jobs.delete(job_id=j.id)
# print("Eval jobs deleted")

In [ ]:
# # Delete namespace (PERMANENT - deletes everything in namespace)
# print("Deleting namespace...")
# nemo.namespaces.delete(NAMESPACE)
# requests.delete(f"{NDS_URL}/v1/datastore/namespaces/{NAMESPACE}")
# print(f"Namespace '{NAMESPACE}' deleted")